In [ ]:
!pip install python-terrier nltk scikit-learn

import pyterrier as pt

if not pt.started():
    pt.init()

import pandas as pd
import re
import nltk

from collections import defaultdict
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

/tmp/ipykernel_19471/619704170.py:5: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


In [ ]:
df = pd.read_csv("oregon_real_estate_with_image_urls.csv")
df["text"] = df["text"].fillna("")

# create document id
df["doc_id"] = df.index.astype(str)

print(df.head())

            type sub_type                                               text  \
0  single_family      NaN  Showings subject to accepted offer. Investor o...   
1  single_family      NaN  *DO NOT DRIVE BY WITHOUT AN APPOINTMENT* Distr...   
2         condos    condo  This exceptional property is priced to sell & ...   
3  single_family      NaN  Commercially zoned property currently utilized...   
4         condos    condo  Sophisticated and beautifully updated riverfro...   

   listPrice    sqft  stories  beds  baths  baths_full  baths_full_calc  \
0   179950.0   698.0      1.0   2.0    1.0         1.0              1.0   
1   399900.0  2736.0      2.0   4.0    2.0         2.0              2.0   
2   220000.0  1130.0      1.0   3.0    3.0         3.0              3.0   
3   230000.0  1293.0      2.0   3.0    2.0         2.0              2.0   
4   899000.0  3172.0      3.0   3.0    4.0         4.0              4.0   

   garage  year_built                                             im

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):


    text = text.lower()


    text = re.sub(r'http\S+|www\S+', '', text)


    text = re.sub(r'[^a-z\s]', '', text)


    tokens = nltk.word_tokenize(text)


    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(tokens)


df["processed_text"] = df["text"].apply(preprocess)

print(df[["text", "processed_text"]].head())

                                                text  \
0  Showings subject to accepted offer. Investor o...   
1  *DO NOT DRIVE BY WITHOUT AN APPOINTMENT* Distr...   
2  This exceptional property is priced to sell & ...   
3  Commercially zoned property currently utilized...   
4  Sophisticated and beautifully updated riverfro...   

                                      processed_text  
0  showing subject accepted offer investor opport...  
1  drive without appointment distressed property ...  
2  exceptional property priced sell mustsee fly f...  
3  commercially zoned property currently utilized...  
4  sophisticated beautifully updated riverfront t...  


In [ ]:
inverted_index=defaultdict(dict)
for doc_id,text in zip(df["doc_id"], df["processed_text"]):
    words = text.split()
    for word in words:
        if doc_id not in inverted_index[word]:
            inverted_index[word][doc_id] = 0
        inverted_index[word][doc_id] += 1
print("Sample Inverted Index:\n")
for i, (term, postings) in enumerate(inverted_index.items()):
    print(term, postings)
    if i == 5:
        break

Sample Inverted Index:

showing {'0': 1, '12': 1, '22': 1, '27': 1, '54': 1, '75': 1, '93': 1, '137': 1, '162': 1, '232': 1, '236': 1, '250': 1, '265': 1, '312': 1, '318': 1, '349': 1, '352': 1, '360': 1, '367': 1, '423': 2, '425': 1, '468': 1, '475': 1, '487': 1, '503': 1, '530': 1, '558': 1, '623': 1, '632': 1, '656': 1, '698': 1, '700': 1, '721': 1, '722': 1, '729': 1, '780': 1, '786': 1, '811': 1, '813': 1, '831': 1, '866': 1, '896': 1, '898': 1, '902': 1, '925': 1, '943': 1, '980': 1, '993': 1, '1036': 1, '1049': 1, '1054': 1, '1072': 1, '1078': 1, '1170': 1, '1181': 1, '1188': 1, '1217': 1, '1239': 1, '1253': 1, '1312': 1, '1340': 1, '1384': 1, '1389': 3, '1401': 1, '1406': 1, '1420': 1, '1450': 1, '1452': 1, '1455': 1, '1458': 1, '1463': 1, '1479': 1, '1493': 1, '1522': 1, '1553': 1, '1564': 1, '1573': 1, '1609': 1, '1625': 1, '1665': 1, '1695': 1, '1716': 1, '1765': 1, '1797': 1, '1824': 1, '1840': 1, '1846': 1, '1862': 1, '1863': 1, '1869': 1, '1875': 1, '1888': 1, '1928': 1, 

In [ ]:
import os

indexer = pt.IterDictIndexer("./index", overwrite=True)
index_ref = indexer.index(df[["doc_id", "processed_text"]].rename(columns={"doc_id": "docno", "processed_text": "text"}).to_dict(orient="records"))

tfidf = pt.terrier.Retriever(
    index_ref,
    wmodel="TF_IDF"
)

18:35:29.494 [ForkJoinPool-2-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (443) - further warnings are suppressed
18:35:40.976 [ForkJoinPool-2-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 20 empty documents


In [ ]:
def search_real_estate(
    query,
    min_price=None,
    max_price=None,
    top_k=5
):

    processed_query = preprocess(query)


    results = tfidf.search(processed_query)


    merged = results.merge(
        df,
        left_on="docno",
        right_on="doc_id"
    )


    if min_price is not None:
        merged = merged[
            merged["listPrice"] >= min_price
        ]

    if max_price is not None:
        merged = merged[
            merged["listPrice"] <= max_price
        ]

    # top k
    merged = merged.head(top_k)

    return merged[
        [
            "doc_id",
            "listPrice",
            "score",
            "text",
            "images" # Changed from 'image_url'
        ]
    ]

# 5. Query Expansion
In this section, we apply relevance feedback principles by expanding the user query with synonyms and related terms to improve recall.

In [ ]:
expansion_map = {
    "garden": ["yard", "landscaping", "outdoor", "flora"],
    "house": ["home", "residence", "dwelling", "property"],
    "beautiful": ["stunning", "gorgeous", "lovely", "exceptional"],
    "modern": ["contemporary", "updated", "renovated"],
    "spacious": ["large", "expansive", "roomy"]
}

def expand_query(query, mapping):
    tokens = query.lower().split()
    expanded_tokens = list(tokens)

    for token in tokens:
        if token in mapping:
            expanded_tokens.extend(mapping[token])

    return " ".join(list(set(expanded_tokens)))


def search_expanded_real_estate(query, min_price=None, max_price=None, top_k=5):

    expanded_q = expand_query(query, expansion_map)
    print(f"Original Query: {query}")
    print(f"Expanded Query: {expanded_q}")


    processed_query = preprocess(expanded_q)


    results = tfidf.search(processed_query)


    merged = results.merge(df, left_on='docno', right_on='doc_id')

    if min_price is not None:
        merged = merged[merged['listPrice'] >= min_price]
    if max_price is not None:
        merged = merged[merged['listPrice'] <= max_price]

    return merged.head(top_k)[['doc_id', 'listPrice', 'score', 'text', 'images']]

### **5.2 Advanced Query Expansion using Embeddings (Word2Vec)**
In this step, we replace the manual synonym mapping with a pre-trained Word2Vec model from the `gensim` library. This model has been trained on a massive corpus (Wikipedia) and can identify semantically related terms automatically.

In [ ]:
!pip install gensim

import gensim.downloader as api

print("Loading Word2Vec model (glove-wiki-gigaword-50)...")
word_vectors = api.load("glove-wiki-gigaword-50")
print("Model loaded successfully!")

def expand_query_with_embeddings(query, model, top_n=3):

    tokens = preprocess(query).split()
    expanded_tokens = list(tokens)

    for token in tokens:
        try:

            similar = [word for word, score in model.most_similar(token, topn=top_n)]
            expanded_tokens.extend(similar)
        except KeyError:

            continue

    return " ".join(list(set(expanded_tokens)))

def search_embedding_expanded_real_estate(query, min_price=None, max_price=None, top_k=5):

    expanded_q = expand_query_with_embeddings(query, word_vectors)
    print(f"Original Query: {query}")
    print(f"Embedding-Expanded Query: {expanded_q}")


    results = tfidf.search(expanded_q)


    merged = results.merge(df, left_on='docno', right_on='doc_id')


    if min_price is not None:
        merged = merged[merged['listPrice'] >= min_price]
    if max_price is not None:
        merged = merged[merged['listPrice'] <= max_price]

    return merged.head(top_k)[['doc_id', 'listPrice', 'score', 'text', 'images']]

Loading Word2Vec model (glove-wiki-gigaword-50)...
Model loaded successfully!


In [ ]:
embedding_results = search_embedding_expanded_real_estate(
    query="modern cottage",
    min_price=150000,
    max_price=600000
)

display(embedding_results)

Original Query: modern cottage
Embedding-Expanded Query: rustic cottage architecture contemporary cottages culture farmhouse modern


,doc_id,listPrice,score,text,images
0,3953,349900.0,16.792844,Welcome to Artisan at Ramona Condominium's. Th...,https://image.pollinations.ai/prompt/A%20high%...
1,8810,354900.0,16.792844,Welcome to Artisan at Ramona Condominium's. Th...,https://image.pollinations.ai/prompt/A%20high%...
2,2688,314900.0,16.592525,Welcome to Artisan at Ramona Condominium's. Th...,https://image.pollinations.ai/prompt/A%20high%...
5,4945,270000.0,13.912903,Modern Luxury Meets Storybook Charm This encha...,https://image.pollinations.ai/prompt/A%20high%...
6,1089,365000.0,12.760616,Tucked away in the serene countryside of Knapp...,https://image.pollinations.ai/prompt/A%20high%...


In [ ]:
expanded_results = search_expanded_real_estate(
    query="beautiful house with garden",
    min_price=200000,
    max_price=900000
)

if not expanded_results.empty:
    display(expanded_results)
else:
    print("No results found for the given criteria.")

Original Query: beautiful house with garden
Expanded Query: property outdoor with stunning garden flora exceptional landscaping home residence beautiful yard dwelling lovely house gorgeous


,doc_id,listPrice,score,text,images
0,10343,799000.0,14.702462,"VERY SECLUDED AND PRIVATE, THIS APPROXIMATELY ...",https://image.pollinations.ai/prompt/A%20high%...
2,3727,425000.0,14.103677,Connect with the picturesque flora and fauna o...,https://image.pollinations.ai/prompt/A%20high%...
3,1185,610000.0,13.952901,"Located on a beautiful, quiet street in Joseph...",https://image.pollinations.ai/prompt/A%20high%...
4,475,889000.0,13.795649,"Stunning, spacious home on nearly an acre with...",https://image.pollinations.ai/prompt/A%20high%...
6,5381,295000.0,13.221965,Gorgeous property in the lovely Logsden area. ...,https://image.pollinations.ai/prompt/A%20high%...


In [ ]:
results = search_expanded_real_estate(
    query="beautiful house with garden",
    min_price=200000,
    max_price=900000,
    top_k=5
)

if not results.empty:
    display(results)
else:
    print("No results found matching those criteria.")

Original Query: beautiful house with garden
Expanded Query: property outdoor with stunning garden flora exceptional landscaping home residence beautiful yard dwelling lovely house gorgeous


,doc_id,listPrice,score,text,images
0,10343,799000.0,14.702462,"VERY SECLUDED AND PRIVATE, THIS APPROXIMATELY ...",https://image.pollinations.ai/prompt/A%20high%...
2,3727,425000.0,14.103677,Connect with the picturesque flora and fauna o...,https://image.pollinations.ai/prompt/A%20high%...
3,1185,610000.0,13.952901,"Located on a beautiful, quiet street in Joseph...",https://image.pollinations.ai/prompt/A%20high%...
4,475,889000.0,13.795649,"Stunning, spacious home on nearly an acre with...",https://image.pollinations.ai/prompt/A%20high%...
6,5381,295000.0,13.221965,Gorgeous property in the lovely Logsden area. ...,https://image.pollinations.ai/prompt/A%20high%...


In [ ]:
from IPython.display import HTML, display

for i, row in results.iterrows():

    print("=" * 70)

    print("Document ID:", row["doc_id"])
    print("Price:", row["listPrice"])
    print("TF-IDF Score:", row["score"])

    print("\nDescription:")
    print(row["text"])

    image_url = row["images"]

    display(
        HTML(
            f'<img src="{image_url}" width="400">'
        )
    )

Document ID: 10343
Price: 799000.0
TF-IDF Score: 14.702461715745425

Description:
VERY SECLUDED AND PRIVATE, THIS APPROXIMATELY 226.07 +/- ACRES OF CURRY COUNTY, "OPPORTUNIY ZONE" TIMBERLAND BOASTS APPROX. 1.[Redacted Entity] Frontage & APPOX. 1 MILE OF FLORAS CREEK FRONTAGE! There are multiple access points off of Floras Creek Rd, with one being a easy to walk Old Logging Road. Wildlife is Abundant, as well as Merchantable and Reproduced Timber. Floras Creek has one of the few Native Coho Salmon Runs left on the Coast! Off-Grid Living? Long-Term Timber Investment? Recreational or LOP Hunting? Properties like this do not come up for Sale Often! Property may qualify for Large Tract Dwelling permit. Timber information available. Curry County Planning office would be a good place to start your Due Diligence.


Document ID: 3727
Price: 425000.0
TF-IDF Score: 14.10367706177253

Description:
Connect with the picturesque flora and fauna of the Columbia River Gorge on this beautiful 19+ acre parcel offering views, calm, and a stunning setting. Tucked in the hills, offering privacy, the property features an old stone structure and a peaceful pond that is one of many natural beauties of this particular parcel. An intentional and mosaic fire fuels reduction project has been completed on the property, creating a more open, maneuverable, and fire-conscious landscape while preserving the property's valuable biodiversity. With a mix of trees, open areas, and potential building sites, this is an ideal setting for a future home and roots or a personal recreational retreat. Enjoy the serenity, scenery, and seclusion while still being within reach of nearby towns and community.


Document ID: 1185
Price: 610000.0
TF-IDF Score: 13.952900526236203

Description:
Located on a beautiful, quiet street in Joseph, Oregon, this custom home sits on a large lot with exceptional curb appeal. The front yard features gorgeous, well-established landscaping, while the fully fenced backyard offers privacy, space, and a designated garden area-perfect for outdoor living and entertaining.Inside, the home offers abundant storage and generous living space throughout. With 4 bedrooms and 2.5 bathrooms, there's plenty of room for family, guests, or a home office setup. The thoughtfully designed layout balances comfort and functionality.The basement adds even more versatility, featuring a spacious additional living area complete with a bar and optional wiring for overhead lighting above a pool table-ideal for a game room, media space, or entertaining hub.This well-maintained property combines space, privacy, and charm in one of Joseph's most desirable settings.


Document ID: 475
Price: 889000.0
TF-IDF Score: 13.795649131521218

Description:
Stunning, spacious home on nearly an acre with sweeping mountain views - ideal for hobbyists, or anyone who loves room to roam. This generous residence offers 6 bedrooms and 3.5 bathrooms, an open-concept main living area that flows effortlessly for entertaining, and multiple family rooms for flexible living. Two convenient laundry rooms and abundant basement storage make daily life and organization easy.Outside, enjoy a fenced yard, an established raspberry garden, and expansive parking for guests, RVs or toys. A large shop provides excellent space for woodworking, car projects, or storage. Quiet, private setting with plenty of outdoor space to garden, play, or take in the vistas. Exceptional blend of functionality and country-style living - must see. Call your agent for a showing today!


Document ID: 5381
Price: 295000.0
TF-IDF Score: 13.221964695027447

Description:
Gorgeous property in the lovely Logsden area. This home has tons of potential, beautiful garden area, RV parking, covered wrap around porch and so much more!! Logsden summers are so nice and you're close to the river, Life is Good in Logsden!


## 6. Performance Evaluation
In this final step, we evaluate the retrieval accuracy and speed of our different search implementations. We will test:
1. **Basic Search** (TF-IDF only)
2. **Manual Expanded Search** (Synonym map)
3. **Embedding Expanded Search** (Word2Vec)

We'll measure the time taken for each and inspect the results for qualitative relevance.

In [ ]:
import time

test_queries = [
    "modern cottage with view",
    "spacious backyard for garden",
    "luxury updated condo near river",
    "cheap fixer upper investment"
]

evaluation_data = []

for q in test_queries:
    start = time.time()
    basic_res = search_real_estate(q, top_k=3)
    basic_time = time.time() - start

    start = time.time()
    manual_res = search_expanded_real_estate(q, top_k=3)
    manual_time = time.time() - start


    start = time.time()
    embedding_res = search_embedding_expanded_real_estate(q, top_k=3)
    embedding_time = time.time() - start

    evaluation_data.append({
        "Query": q,
        "Basic Time (s)": round(basic_time, 4),
        "Manual Time (s)": round(manual_time, 4),
        "Embedding Time (s)": round(embedding_time, 4),
        "Expansion Count (Emb)": len(expand_query_with_embeddings(q, word_vectors).split())
    })

eval_df = pd.DataFrame(evaluation_data)
display(eval_df)

print("\n--- Detailed Results for 'luxury updated condo near river' (Embedding Search) ---")
display(search_embedding_expanded_real_estate("luxury updated condo near river", top_k=3))

Original Query: modern cottage with view
Expanded Query: with view cottage updated contemporary renovated modern
Original Query: modern cottage with view
Embedding-Expanded Query: rustic views view cottage architecture beyond contemporary cottages viewed culture farmhouse modern
Original Query: spacious backyard for garden
Expanded Query: outdoor garden for roomy flora landscaping spacious backyard large yard expansive
Original Query: spacious backyard for garden
Embedding-Expanded Query: fountain lawn playground garden gardens picnic rooms spacious backyard barbecues furnished cramped
Original Query: luxury updated condo near river
Expanded Query: near updated luxury river condo
Original Query: luxury updated condo near river
Embedding-Expanded Query: condominiums town river tributary updated creek hotels condos nearby rivers edition update condominium luxury condo editions near expensive area luxurious
Original Query: cheap fixer upper investment
Expanded Query: investment cheap uppe

,Query,Basic Time (s),Manual Time (s),Embedding Time (s),Expansion Count (Emb)
0,modern cottage with view,0.0676,0.0559,0.1831,12
1,spacious backyard for garden,0.1237,0.1155,0.1648,12
2,luxury updated condo near river,0.0566,0.0755,0.2391,20
3,cheap fixer upper investment,0.0494,0.0512,0.2440,16



--- Detailed Results for 'luxury updated condo near river' (Embedding Search) ---
Original Query: luxury updated condo near river
Embedding-Expanded Query: condominiums town river tributary updated creek hotels condos nearby rivers edition update condominium luxury condo editions near expensive area luxurious


,doc_id,listPrice,score,text,images
0,5701,235000.0,24.023061,Escape to peaceful riverfront living in this c...,https://image.pollinations.ai/prompt/A%20high%...
1,6250,879000.0,22.017149,Remodeled Six Plex with beautiful Columbia Riv...,https://image.pollinations.ai/prompt/A%20high%...
2,1350,259000.0,20.675951,Condo is on the ground level for easy access a...,https://image.pollinations.ai/prompt/A%20high%...
